# SU(2) Paper Benchmark Suite

This notebook is the paper-style benchmark entry point. It keeps the setup fixed:

```text
H, t -> Hamiltonian-conditioned circuit-token diffusion on SU(2)^15
     -> local SU(2) refinement in the line-4cz template
```

Default behavior: run a short sanity benchmark. For paper numbers, flip `RUN_FULL_PAPER_BENCHMARK = True` near the bottom.

In [ ]:
BRANCH = "codex/paper-benchmark-suite"
!pip install -q --force-reinstall --no-deps git+https://github.com/joe-singh/su2diffusion.git@{BRANCH}

In [ ]:
from dataclasses import replace

import torch

from su2diffusion import (
    center_names_for_config,
    get_circuit_experiment_config,
    get_experiment_config,
    get_paper_benchmark_config,
    paper_benchmark_summary_rows,
    plot_paper_benchmark_suite,
    print_paper_benchmark_suite,
    run_experiment,
    run_paper_benchmark_suite,
    save_paper_benchmark_artifacts,
)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Train the local SU(2) gate pool

This is the same local single-qubit generator used in the demo notebook. The paper benchmark compares its generated-search baseline against the Hamiltonian-conditioned circuit-token proposal.

In [ ]:
LOCAL_CONFIG_NAME = "baseline-clifford-cond"
LOCAL_SAMPLE_COUNT = 1000

local_config = get_experiment_config(LOCAL_CONFIG_NAME)
local_config = replace(local_config, sample_count=LOCAL_SAMPLE_COUNT, reference_count=LOCAL_SAMPLE_COUNT)

local_result = run_experiment(local_config, device=device)
center_names = center_names_for_config(local_result.config.data)
local_gates = local_result.generated_stochastic
local_labels = [center_names[int(label)] for label in local_result.stochastic_labels]

print(f"trained local config: {local_result.config.name}")
print(f"generated local gates: {tuple(local_gates.shape)}")

## Shared circuit-token model settings

The template is the current 3-qubit line with four CZs:

```text
L0 - CZ01 - L1 - CZ12 - L2 - CZ01 - L3 - CZ12 - L4
```

That means five local layers over three qubits, or `SU(2)^15`.

In [ ]:
base_circuit_config = get_circuit_experiment_config("medium-circuit-near-clifford")
paper_circuit_config = replace(
    base_circuit_config,
    name="paper-hamiltonian-3q-line4cz-token",
    sample_count=256,
    n_slots=15,
    train=replace(
        base_circuit_config.train,
        batch_size=256,
        hidden=256,
    ),
)
print(paper_circuit_config)

## Smoke benchmark

This is a tiny branch-test version. It checks that the benchmark machinery works without producing meaningful paper numbers. Use the full benchmark cell for actual results.

In [ ]:
quick_benchmark_config = get_paper_benchmark_config("smoke")

print(quick_benchmark_config)

quick_paper_benchmark = run_paper_benchmark_suite(
    generated_gates=local_gates,
    generated_labels=local_labels,
    circuit_config=paper_circuit_config,
    benchmark_config=quick_benchmark_config,
    device=device,
    show_progress=True,
)

print_paper_benchmark_suite(quick_paper_benchmark)
plot_paper_benchmark_suite(quick_paper_benchmark)

quick_artifacts = save_paper_benchmark_artifacts(
    quick_paper_benchmark,
    "paper_artifacts/quick",
)
print("\nsaved artifacts:")
for name, path in quick_artifacts.items():
    print(f"  {name}: {path}")

## Full paper benchmark

Turn this on when you want the current headline table. It reruns training/refinement across three independent seeds and 48 held-out Hamiltonians per seed.

In [ ]:
RUN_FULL_PAPER_BENCHMARK = False

if RUN_FULL_PAPER_BENCHMARK:
    full_benchmark_config = get_paper_benchmark_config("level3")
    full_paper_benchmark = run_paper_benchmark_suite(
        generated_gates=local_gates,
        generated_labels=local_labels,
        circuit_config=paper_circuit_config,
        benchmark_config=full_benchmark_config,
        device=device,
        show_progress=True,
    )

    print_paper_benchmark_suite(full_paper_benchmark)
    plot_paper_benchmark_suite(full_paper_benchmark)

    full_artifacts = save_paper_benchmark_artifacts(
        full_paper_benchmark,
        "paper_artifacts/full",
    )
    print("\nsaved artifacts:")
    for name, path in full_artifacts.items():
        print(f"  {name}: {path}")
else:
    print("Full paper benchmark skipped. Set RUN_FULL_PAPER_BENCHMARK = True to run it.")

## Programmatic summary rows

Use this if you want to paste summary rows into notes, a draft, or a spreadsheet.

In [ ]:
for row in paper_benchmark_summary_rows(quick_paper_benchmark):
    print(row)